# Meeting 11: the class notebook

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 11.

This notebook holds the two Colab blocks of the meeting, one for Act 1 and one for Act 2. The slides explain the ideas; here your group runs them on its own image and posts what it finds. The **full notebooks** of the three acts, linked on the meeting's page, explain every step and hold the work for after the meeting.

## How this works

- **First, before anything else, ask Colab for a GPU.** Open the **Runtime** menu, choose **Change runtime type**, select **T4 GPU** under *Hardware accelerator*, and choose **Save**. Colab erases the work of a notebook when it changes its machine, so this comes first. If Colab answers that no GPU is available, go on without one: every cell of this notebook also runs on a machine without a GPU, and one GPU in a group is enough.
- **Then**, at the start of the meeting, choose **Save a copy in Drive** in the **File** menu, run the first three code cells (the GPU check, the setup, your group's number), and leave the notebook open: the first block comes after the first slides.
- **Work in a group of two or three.** Every member runs their own copy. Use the number that your group had in meeting 10, so that you work on the same image; a group without a number chooses one from 1 to 30.
- **Two blocks**, one for Act 1 and one for Act 2. A block has a few cells to run, and one thing to write or to change.
- **Two posts** on the discussion board of this meeting, one in each block. **One member posts for the group**, and it need not be the same member each time. **Every post begins with the names of all the members of your group.** The second post is a reply to the first. Under every image, write one sentence that says what the image shows.
- **Nothing is uploaded.** The posts are the record of your group's work. **Review Quiz 11** opens when the meeting ends; its questions name the steps of the full notebooks.

**Your turn.** Double-click this cell and replace this line with the names of the members of your group.

In [ ]:
# Run this cell first. It says whether this session has a GPU.
import torch

print("PyTorch version:", torch.__version__)
print("Does PyTorch see a GPU?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("The GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Open the Runtime menu, choose Change runtime type, select T4 GPU, choose Save, and run this cell again.")

In [ ]:
# Setup. Run this cell after the GPU check.
import os
import time
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

# The address of the course's files for this unit, and of the folder that holds the images.
SITE = "https://ikoutis.github.io/cs301-labs/unit-2/"
IMAGES = SITE + "images/"

GPU = torch.cuda.is_available()          # True when this session has a GPU


def read_image(file):
    """Read an image file into a NumPy array with values from 0 to 1."""
    return np.array(Image.open(urlopen(IMAGES + file))) / 255.0


def show(pictures, titles):
    """Draw arrays as pictures: an array with two dimensions in grey, one with three in colour."""
    fig, axes = plt.subplots(1, len(pictures), figsize=(5 * len(pictures), 4.2), squeeze=False)
    for ax, picture, title in zip(axes.flat, pictures, titles):
        ax.imshow(np.clip(picture, 0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


def show_colours(colours, names, title):
    """Draw colours as a row of squares, each under its name."""
    fig, axes = plt.subplots(1, len(colours), figsize=(1.7 * len(colours), 2.0), squeeze=False)
    for ax, colour, name in zip(axes.flat, colours, names):
        ax.imshow(np.clip(colour, 0, 1).reshape(1, 1, 3))
        ax.set_title(name, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.show()


def reference_colours(X, K=5, rounds=10):
    """K colours that occur often among the rows of X, from the darkest to the lightest."""
    grey_weights = np.array([0.299, 0.587, 0.114])
    sample = X[::7]                                    # every seventh point is enough
    order = np.argsort(sample @ grey_weights)          # the points, from the darkest to the lightest
    C = np.array([sample[part].mean(axis=0) for part in np.array_split(order, K)])
    for _ in range(rounds):                            # a few rounds of the method k-means
        distance = ((sample[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
        nearest = distance.argmin(axis=1)
        for j in range(K):
            if (nearest == j).any():
                C[j] = sample[nearest == j].mean(axis=0)
    return C[np.argsort(C @ grey_weights)]


print("Setup is done. This session", "has a GPU." if GPU else "has no GPU.")

**Your turn.** Type your group's number in the next cell, in place of the `0`, and run it: the number that your group had in meeting 10, so that you work on the same image.

In [ ]:
GROUP = 0        # replace 0 with your group's number, from 1 to 30

assert 1 <= GROUP <= 30, "Type your group's number in place of the 0."

images = pd.read_csv(IMAGES + "images.csv", index_col="number")
picture_info = images.loc[GROUP]
image = read_image(picture_info["file"])
height, width = image.shape[0], image.shape[1]
print(f"Group {GROUP}. Image: {picture_info['title']} ({picture_info['creator']}), {height} rows and {width} columns of pixels.")
show([image], [f"image {GROUP}"])

## <font color="#1967d2"><b>Act 1: A layer of neurons is a matrix</b></font>

The points are the pixels of your image: `X = image.reshape(-1, 3)` has one row for each pixel, with its red, its green and its blue number, as the days of Newark airport were the rows of a matrix on the slides. The next cell builds the layer of the slides, five neurons, one for each of five reference colours of your image, and defines `layer_loops`, which computes the scores of all the neurons on all the points with three loops.

In [ ]:
X = image.reshape(-1, 3)                 # one row for each pixel: n points with m = 3 coordinates
n, m = X.shape
K = 5
C = reference_colours(X, K)              # K rows and 3 columns: the reference colours, darkest first
W = C.copy()                             # the weights of the layer: row j is the colour of neuron j
b = -0.5 * (C ** 2).sum(axis=1)          # the biases: minus half the sum of the squares of each colour
print(f"X: {X.shape}, which is n = {n:,} points with m = {m} coordinates.  W: {W.shape}.  b: {b.shape}.")
show_colours(C, [f"neuron {j}" for j in range(K)], "The reference colours: row j of W")


def layer_loops(X, W, b):
    """The scores of the K neurons of a layer on the n points of X, one multiplication at a time."""
    n, m = X.shape
    K = W.shape[0]
    S = np.empty((n, K))
    for i in range(n):                   # every point
        for j in range(K):               # every neuron
            total = b[j]
            for c in range(m):           # every coordinate
                total += W[j, c] * X[i, c]
            S[i, j] = total
    return S


S_first = layer_loops(X[:2000], W, b)    # the three loops, on the first 2,000 points only
print("shape of the scores of the first 2,000 points:", S_first.shape)

**Your turn.** The three loops are one product of matrices, as for the four neurons and the days of the slides. Complete the function `layer` in the next cell: replace the three dots with **one line** that uses `@` and no loop. `X` has shape (n, m) and `W` has shape (K, m); the result must have n rows and K columns, one for each neuron. Mind which array has to be transposed, and add the bias. Then run the cell after it, which checks your line against the three loops.

In [ ]:
def layer(X, W, b):
    """The scores of the K neurons of a layer on the n points of X: an array with n rows and K columns."""
    return ...      # Your turn: replace the three dots with one line that uses @

In [ ]:
try:
    S = layer(X, W, b)
except ValueError:
    raise AssertionError(
        "The shapes do not fit. X has shape (n, 3) and W has shape (K, 3), and the result must have n rows "
        "and K columns. Which array has to be transposed?") from None
assert S is not ..., "Replace the three dots in the function layer with your line, and run that cell again."
S = np.asarray(S)
assert S.shape == (n, K), (
    f"layer returned an array of shape {S.shape}. It must be {(n, K)}: one row for each point and one column for each neuron.")
assert np.allclose(S[:2000], S_first), (
    "The scores differ from those of layer_loops. Check which array is transposed, and that the bias is added.")
print(f"Correct: layer(X, W, b) has shape {S.shape} and agrees with the three loops.")

start = time.perf_counter()
layer_loops(X, W, b)
t_loops = time.perf_counter() - start
start = time.perf_counter()
for _ in range(20):
    layer(X, W, b)
t_product = (time.perf_counter() - start) / 20
print(f"On all {n:,} points: three loops {t_loops:.3f} s, one product {t_product:.4f} s. "
      f"The loops take {t_loops / t_product:.0f} times as long as the product.")

winner = S.argmax(axis=1)                # for each point, the neuron with the largest score
show([image, C[winner].reshape(height, width, 3)], ["the image", f"every pixel in the colour of its neuron: {K} colours"])

**Your turn.** The next cell repaints your image with the two lines of the slides: the softmax turns the scores of every pixel into five probabilities, and a second product mixes five paints, one for each neuron, in those proportions. Replace the five paints with colours of your choice, three numbers from 0 to 1 for red, green and blue, and run the cell with the temperatures 1, 0.1 and 0.002 before you settle on one.

In [ ]:
def softmax_rows(S):
    """The softmax of every row of S: positive numbers that add to 1 along each row."""
    shifted = S - S.max(axis=1, keepdims=True)       # the largest entry of every row becomes 0
    E = np.exp(shifted)
    return E / E.sum(axis=1, keepdims=True)


PAINT = np.array([[0.10, 0.07, 0.30],      # the paint of neuron 0, the neuron of the darkest reference colour
                  [0.80, 0.10, 0.35],      # the paint of neuron 1
                  [0.98, 0.55, 0.15],      # the paint of neuron 2
                  [0.99, 0.85, 0.40],      # the paint of neuron 3
                  [1.00, 1.00, 0.92]])     # the paint of neuron 4, the neuron of the lightest reference colour
T = 0.02                                   # the temperature of the softmax: try 1, 0.1 and 0.002

assert PAINT.shape == (K, 3), f"PAINT must have {K} rows, one for each neuron, and 3 columns."
Prob = softmax_rows(layer(X, W, b) / T)    # scores, then probabilities: n rows and K columns
new = Prob @ PAINT                         # probabilities, then colours: n rows and 3 columns
show_colours(PAINT, [f"paint {j}" for j in range(K)], "The paints: row j of PAINT is the paint of neuron j")
show([image, new.reshape(height, width, 3)], ["the image", f"repainted, at temperature {T}"])

paints = "; ".join(str(tuple(float(v) for v in np.round(paint, 2))) for paint in PAINT)
print("Line for the discussion board:")
print(f"Image {GROUP} ({picture_info['title']}), repainted with {K} paints at temperature {T}. "
      f"Paints (red, green, blue): {paints}.")

**Post now.** One member posts for the group on the discussion board of this meeting. Begin the post with the names of all the members of your group. The post contains:

- the line for the discussion board that the cell printed;
- the picture of your image beside its repainted version. To save a figure, right-click it and choose **Save image as**;
- under the picture, one sentence that says what it shows;
- one sentence on what the temperature changed.

## <font color="#1967d2"><b>Act 2: The same layer in PyTorch, and on a GPU</b></font>

The next cell writes the layer in PyTorch. `torch.from_numpy` turns an array into a tensor, `.float()` makes its numbers `float32`, and the line `X_t @ W_t.T + b_t` is the line of Act 1. If this session has a GPU, the cell then copies the three tensors into the memory of the GPU with `.to("cuda")` and computes the scores there.

In [ ]:
X_t = torch.from_numpy(X).float()        # the points, as a tensor of float32 numbers
W_t = torch.from_numpy(W).float()        # the weights of the layer
b_t = torch.from_numpy(b).float()        # the biases
S_t = X_t @ W_t.T + b_t                  # the line of Act 1, unchanged
print("X_t:", tuple(X_t.shape), X_t.dtype, "  S_t:", tuple(S_t.shape), "  on the device:", X_t.device)
print(f"largest difference from the scores of the three loops: {np.abs(S_t[:2000].numpy() - S_first).max():.1e}"
      "  (float32 against float64)")

if GPU:
    X_g, W_g, b_g = X_t.to("cuda"), W_t.to("cuda"), b_t.to("cuda")     # copies, in the memory of the GPU
    S_g = X_g @ W_g.T + b_g              # the same line, carried out by the GPU
    print("X_g is on the device:", X_g.device, "  S_g is on the device:", S_g.device)
    print(f"largest difference between the scores of the GPU and of the CPU: {float((S_g.cpu() - S_t).abs().max()):.1e}")
else:
    print("This session has no GPU, so every tensor stays in the memory of the CPU.")

**Your turn.** The next cell measures `X @ W.T + b` for n = m = K = size, at seven sizes from 64 to 4,096, on the CPU and, if this session has one, on the GPU. It takes up to half a minute. Then answer, from the table: at which sizes is the CPU the faster one, and at which the GPU? From size 2,048 to 4,096, by what factor does the time of the CPU grow?

In [ ]:
def finish(device):
    """Wait until the device has finished its work. A GPU may still be working when PyTorch goes on."""
    if device == "cuda":
        torch.cuda.synchronize()


def time_product(size, device):
    """The average time, in seconds, of X @ W.T + b on the device, with n = m = K = size."""
    X = torch.rand(size, size, device=device)        # random float32 numbers, made on the device
    W = torch.rand(size, size, device=device)
    b = torch.rand(size, device=device)
    repeats = max(1, min(200, 8_000_000_000 // size ** 3))     # many repeats when the product is small
    finish(device)
    start = time.perf_counter()
    for _ in range(repeats):
        S = X @ W.T + b
    finish(device)
    return (time.perf_counter() - start) / repeats


sizes = [64, 128, 256, 512, 1024, 2048, 4096]
devices = ["cpu", "cuda"] if GPU else ["cpu"]
seconds = {}
for device in devices:
    time_product(64, device)             # one small product that is not measured: the first use of a device is slow
    seconds[device] = [time_product(size, device) for size in sizes]

gpu_name = torch.cuda.get_device_name(0) if GPU else "none"
print(f"X @ W.T + b with n = m = K = size, in seconds. CPU: {os.cpu_count()} logical cores. GPU: {gpu_name}.")
print(f"{'size':>6s}{'multiplications':>18s}{'CPU':>12s}" + (f"{'GPU':>12s}{'CPU / GPU':>12s}" if GPU else ""))
for i, size in enumerate(sizes):
    line = f"{size:6d}{size ** 3:18,d}{seconds['cpu'][i]:12.6f}"
    if GPU:
        line += f"{seconds['cuda'][i]:12.6f}{seconds['cpu'][i] / seconds['cuda'][i]:12.1f}"
    print(line)

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(sizes, seconds["cpu"], marker="o", color="#2a78d6", label="CPU")
ax.annotate("CPU", (sizes[-1], seconds["cpu"][-1]), xytext=(8, 0), textcoords="offset points", va="center")
if GPU:
    ax.plot(sizes, seconds["cuda"], marker="s", linestyle="--", color="#eb6834", label="GPU")
    ax.annotate("GPU", (sizes[-1], seconds["cuda"][-1]), xytext=(8, 0), textcoords="offset points", va="center")
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(sizes)
ax.set_xticklabels([str(size) for size in sizes])
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda value, position: f"{value:g}"))
ax.margins(x=0.08)
ax.set_xlabel("size: the number of points, of coordinates and of neurons")
ax.set_ylabel("seconds for one product")
ax.set_title("The time of X @ W.T + b. Each step up the axis is 10 times longer.")
ax.grid(True, which="major", color="#e1e0d9", linewidth=0.8)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

last, first = len(sizes) - 1, 0
print("Line for the discussion board:")
if GPU:
    print(f"GPU {gpu_name}, CPU with {os.cpu_count()} logical cores. Size 4096: CPU {seconds['cpu'][last]:.3f} s, "
          f"GPU {seconds['cuda'][last]:.4f} s, so the GPU is {seconds['cpu'][last] / seconds['cuda'][last]:.0f} times as fast. "
          f"Size 64: CPU {seconds['cpu'][first] * 1e6:.0f} microseconds, GPU {seconds['cuda'][first] * 1e6:.0f} microseconds.")
else:
    print(f"No GPU in this session. CPU with {os.cpu_count()} logical cores. Size 4096: {seconds['cpu'][last]:.3f} s. "
          f"Size 64: {seconds['cpu'][first] * 1e6:.0f} microseconds.")

**Post now**, as a reply to your group's first post. One member posts for the group; begin with the names of all its members. The post contains the line that the cell printed under the chart; the chart, with one sentence that says what it shows; your two answers; and the name of the GPU, if your session had one.

## After the meeting

- **The full notebooks** of the two acts explain every step and hold the work for after the meeting, with checks that test your code: [Act 1](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-11-act-1.ipynb), [Act 2](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-11-act-2.ipynb).
- **Review Quiz 11** on Canvas opens when the meeting ends. Its questions name the steps of the full notebooks that they draw on.
- Your copy of this notebook stays in your Google Drive, in the folder *Colab Notebooks*.